# Evaluation Quick Start

This notebook will allow you to experiment with the prediction task and use CIDER to evaluate LMs' capability of individual-level privacy understanding.

1. Preview example prompts for each evaluation setup.
2. Run model evaluation and write prediction outputs.
3. Analyze prediction results across models, setups, and ICL settings.
4. (Optional) Measure prompt sensitivity across paraphrased HC prompts.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "evaluation" / "configs").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "evaluation" / "helper"))
sys.path.insert(0, str(ROOT / "evaluation"))

CONFIG = ROOT / "evaluation" / "configs" / "sample.json"
CONFIG

## 1. Preview example prompts


In [ ]:
from config import load_json_config, run_subdir
from load_data import load_user_data
from prompts.preview import (
    build_example_prompts,
    display_example_prompts,
    save_example_prompts,
)

cfg = load_json_config(CONFIG)
setups = sorted({s for m in cfg["models"] for s in m["setups"]})
print(f"Selected setups from config: {setups}")
print(f"Batch: {cfg['batch_id']}  k={cfg['dataset']['n_icl_examples']}  "
      f"sample_frac={cfg['dataset']['sample_frac']}")

user_data = load_user_data(cfg)
examples = build_example_prompts(
    user_data,
    setups,
    variant_shuffle_seed=int(cfg["dataset"]["seed"]),
)


display_example_prompts(examples)

out_dir = Path(cfg["output_root"]) / "prompts" / run_subdir(cfg)
saved = save_example_prompts(examples, out_dir / "example_prompts.txt")
print(f"Also saved plain-text copy → {saved}")


## 2. Model evaluation

Calls the model for each setup in `sample.json` and writes predictions under:`<output_root>/<model>/<batch_id>/k{n_icl_examples}_pr{icl_reference_k}/<setup>.jsonl`



In [ ]:
import subprocess

cmd = [
    sys.executable,
    "evaluation/model_eval/run_eval.py",
    "--config",
    str(CONFIG),
]
print("+", " ".join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


## 3. Analyze Prediction Results

Score every model under `outputs/existing/<model>/archived/k{N}_pr{R}/` for **all ks** and setups (`H`, `HL`, `HC`).


In [ ]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

runs_root = ROOT / "outputs" / "existing" # change to your own output root
batch_id = "archived" # change to your own output batch_id
ks = [1, 4, 5, 6]
pr = 6
analysis_dir = runs_root / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

analysis_scripts = [
    "boundary_level_accuracy.py",
    "variant_level_fp_fn.py",
]

for script in analysis_scripts:
    cmd = [
        sys.executable,
        str(ROOT / "evaluation" / "analysis" / script),
        "--runs-root",
        str(runs_root),
        "--batch-id",
        batch_id,
        "--ks",
        ",".join(str(k) for k in ks),
        "--pr",
        str(pr),
    ]
    print("+", " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=ROOT, check=True)


def _show_csvs(pattern: str, title: str, head: int | None = None) -> None:
    paths = sorted(analysis_dir.glob(pattern))

    if not paths:
        print(f"No {title.lower()} found under {analysis_dir}")
        return

    display(Markdown(f"### {title}"))

    for path in paths:
        display(Markdown(f"**`{path.name}`**"))
        df = pd.read_csv(path)
        display(df if head is None else df.head(head))


def _show_figures(pattern: str, title: str) -> None:
    paths = sorted(analysis_dir.glob(pattern))

    if not paths:
        print(f"No {title.lower()} found under {analysis_dir}")
        return

    display(Markdown(f"### {title}"))

    for path in paths:
        display(Markdown(f"**`{path.name}`**"))
        display(Image(filename=str(path), width=720))


_show_csvs(
    "accuracy_*.csv",
    "Boundary-level accuracy tables",
)

_show_csvs(
    "variant_fp_fn_rates_*.csv",
    "Variant-level FP/FN tables",
    head=20,
)

_show_figures(
    "accuracy_setups_k*.png",
    "Accuracy across setups",
)

_show_figures(
    "accuracy_by_k_*.png",
    "Accuracy across k",
)

_show_figures(
    "fp_fn_shift_*.png",
    "FP/FN shift figures",
)


## 4. Prompt sensitivity (Optional)

Measure how stable **HC** predictions are across paraphrased prompts (`v0` / `v1` / `v2`).
To score existing runs without new API calls, add `--metrics-only` to the command below.


In [ ]:
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from config import load_json_config
from constants import REPO_ROOT

cfg = load_json_config(CONFIG)
variants = cfg.get("prompt_variants") or ["v0", "v1", "v2"]
print(f"Prompt variants: {variants}")
print("Setup forced to HC by run_prompt_sensitivity.py")

cmd = [
    sys.executable,
    "evaluation/model_eval/run_prompt_sensitivity.py",
    "--config",
    str(CONFIG),
]
print("+", " ".join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)

sens_root = Path(cfg["output_root"])
if sens_root.name == "eval":
    sens_root = REPO_ROOT / "outputs" / "prompt_sensitivity"
k = int(cfg["dataset"]["n_icl_examples"])
out_dir = sens_root / "analysis_prompt_sensitivity" / f"{cfg['batch_id']}_k{k}"
print(f"Sensitivity tables → {out_dir}")

for name in (
    "metrics_by_variant.csv",
    "sensitivity_summary.csv",
    "pairwise_prediction_agreement.csv",
):
    path = out_dir / name
    if not path.exists():
        print(f"Missing: {path}")
        continue
    display(Markdown(f"**`{path.name}`**"))
    display(pd.read_csv(path))
